In [ ]:
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try: os.unlink(pth)
        except OSError: pass
tmp = Path('/tmp/wheelhouse'); tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    t = tmp / name
    if not t.exists(): os.symlink(w, t)
wheels = sorted(str(w) for w in tmp.glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('installed', len(wheels), 'wheels')

In [ ]:
MAKE_ADAPTERS_SRC = "\"\"\"Turn the raw PEFT output of train_g1.py into scorer-format adapter directories and verify them against the sample adapter format.\n\nVariants written under <out>/:\n  g1_full/    all trained LoRA matrices (attention + MLP)\n  g1_attn/    attention-only subset (q,k,v,o): the safest set for vLLM LoRA support\n  g1_canary/  attention-only with RANDOM lora_B (so the adapter visibly changes outputs even after only a few training steps):\n              used only to prove the serving path really applies adapters (\"silently zero\" LoRA was reported by other teams)\n\nEvery variant: bf16 tensors, keys `base_model.model.model.language_model.layers.N.<module>.lora_{A,B}.weight`, adapter_config.json with\nlist-valued target_modules and base_model_name_or_path = the scorer's model id.\n\nUsage: python make_adapters.py <adapter_full_dir> <out_dir> [sample_adapter_dir]\n\"\"\"\n\nimport json\nimport re\nimport sys\nfrom pathlib import Path\n\nimport torch\nfrom safetensors.torch import load_file, save_file\n\nimport os\nSCORER_MODEL = os.environ.get(\"SCORER_MODEL\", \"google/gemma-4-31b-it-qat-w4a16-ct\")   # e.g. google/gemma-4-e4b-it-qat-w4a16-ct\nKEY_RE = re.compile(r\"^base_model\\.model\\.model\\.language_model\\.layers\\.(\\d+)\\.(self_attn\\.(q_proj|k_proj|v_proj|o_proj)|mlp\\.(gate_proj|up_proj|down_proj))\\.lora_(A|B)\\.weight$\")\nATTN = (\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\")\n\n\ndef write_variant(out: Path, tensors: dict[str, torch.Tensor], cfg: dict) -> dict:\n    out.mkdir(parents=True, exist_ok=True)\n    mods = sorted({k.split(\".\")[-3] for k in tensors})\n    cfg = dict(cfg, base_model_name_or_path=SCORER_MODEL, inference_mode=True, target_modules=mods)\n    cfg.pop(\"layers_to_transform\", None) if cfg.get(\"layers_to_transform\") is None else None\n    (out / \"adapter_config.json\").write_text(json.dumps(cfg, indent=2), encoding=\"utf-8\")\n    save_file({k: v.to(torch.bfloat16).contiguous() for k, v in tensors.items()}, str(out / \"adapter_model.safetensors\"), metadata={\"format\": \"pt\"})\n    size = (out / \"adapter_model.safetensors\").stat().st_size\n    return {\"dir\": str(out), \"tensors\": len(tensors), \"target_modules\": mods, \"bytes\": size}\n\n\ndef main(src: Path, out: Path, sample: Path | None) -> dict:\n    t = load_file(str(src / \"adapter_model.safetensors\"))\n    cfg = json.loads((src / \"adapter_config.json\").read_text(encoding=\"utf-8\"))\n    bad = [k for k in t if not KEY_RE.match(k)]\n    assert not bad, f\"unexpected tensor names, e.g. {bad[:3]}\"\n    report: dict = {\"n_tensors\": len(t), \"rank\": cfg.get(\"r\"), \"alpha\": cfg.get(\"lora_alpha\"), \"dtype_in\": str(next(iter(t.values())).dtype)}\n    if sample is not None and (sample / \"adapter_model.safetensors\").exists():\n        sk = list(load_file(str(sample / \"adapter_model.safetensors\")))\n        report[\"sample_key_example\"] = sk[0]\n        report[\"our_key_example\"] = sorted(t)[0]\n        # same naming scheme (prefix up to the layer index) as the scorer's sample adapter\n        pre = lambda k: k.split(\".layers.\")[0]  # noqa: E731\n        assert pre(sk[0]) == pre(sorted(t)[0]), (pre(sk[0]), pre(sorted(t)[0]))\n        sample_cfg = json.loads((sample / \"adapter_config.json\").read_text(encoding=\"utf-8\"))\n        report[\"sample_cfg_keys_missing_in_ours\"] = sorted(set(sample_cfg) - set(cfg))\n    report[\"g1_full\"] = write_variant(out / \"g1_full\", t, cfg)\n    attn = {k: v for k, v in t.items() if k.split(\".\")[-3] in ATTN}\n    report[\"g1_attn\"] = write_variant(out / \"g1_attn\", attn, cfg)\n    g = torch.Generator().manual_seed(0)\n    canary = {k: (torch.randn(v.shape, generator=g) * 0.02 if k.endswith(\"lora_B.weight\") else v.clone()) for k, v in attn.items()}\n    report[\"g1_canary\"] = write_variant(out / \"g1_canary\", canary, cfg)\n    (out / \"adapters_report.json\").write_text(json.dumps(report, indent=1), encoding=\"utf-8\")\n    return report\n\n\nif __name__ == \"__main__\":\n    r = main(Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3]) if len(sys.argv) > 3 else None)\n    print(json.dumps(r, indent=1))\n"

In [ ]:
import glob, json, os, re, sys, time
sys.modules["torchao"] = None   # Kaggle's torchao 0.10.0 is rejected by peft 0.19.1
from pathlib import Path
import torch, transformers
print('torch', torch.__version__, 'transformers', transformers.__version__, 'cpus', os.cpu_count())
cand = sorted(glob.glob('/kaggle/input/models/google/gemma-4/*/gemma-4-e4b-it-qat-q4_0-unquantized/*/config.json'))
print('model candidates:', cand); assert len(cand) == 1
MODEL = str(Path(cand[0]).parent)
cfg = transformers.AutoConfig.from_pretrained(MODEL)
tc = cfg.get_text_config()
facts = {k: getattr(tc, k, None) for k in ('hidden_size', 'num_hidden_layers', 'num_attention_heads', 'num_key_value_heads', 'head_dim', 'global_head_dim', 'sliding_window',
         'hidden_size_per_layer_input', 'vocab_size_per_layer_input', 'num_kv_shared_layers', 'use_double_wide_mlp', 'attention_k_eq_v', 'final_logit_softcapping', 'vocab_size',
         'intermediate_size', 'enable_moe_block', 'tie_word_embeddings')}
import collections
facts['layer_types'] = dict(collections.Counter(tc.layer_types)); facts['architectures'] = cfg.architectures; facts['tie_word_embeddings_top'] = getattr(cfg, 'tie_word_embeddings', None)
print('TEXT CONFIG:', json.dumps(facts, indent=1))
t0 = time.time()
model = transformers.Gemma4ForConditionalGeneration.from_pretrained(MODEL, dtype=torch.bfloat16, attn_implementation='sdpa')
print('loaded in', round(time.time() - t0), 's | params (B):', round(sum(p.numel() for p in model.parameters()) / 1e9, 2))
from peft import LoraConfig, get_peft_model
TARGET_RE = r".*language_model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))"
model.config.use_cache = False
pm = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias='none', target_modules=TARGET_RE, task_type='CAUSAL_LM'))
mods = [n for n, _ in pm.named_modules() if n.endswith('.lora_A.default')]
by = collections.Counter(n.split('.lora_A')[0].split('.')[-1] for n in mods)
print('LoRA modules:', len(mods), dict(by), '| trainable params (M):', round(sum(p.numel() for p in pm.parameters() if p.requires_grad) / 1e6, 1))
print('example module names:', mods[:2], mods[-1])
# one forward + backward with the chunked lm_head loss on a short random sequence (CPU)
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
pm.train(); model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
ids = torch.randint(5, 20000, (1, 192))
t0 = time.time()
hs = model.model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False).last_hidden_state[0]
softcap = tc.final_logit_softcapping
def chunk_loss(h, tgt):
    lg = model.lm_head(h).float()
    if softcap: lg = torch.tanh(lg / softcap) * softcap
    return F.cross_entropy(lg, tgt, reduction='sum')
pos = list(range(50, 190)); tgt = ids[0, [p + 1 for p in pos]]
loss = checkpoint(chunk_loss, hs[pos], tgt, use_reentrant=False) / len(pos)
loss.backward()
gn = [float(p.grad.norm()) for p in pm.parameters() if p.requires_grad and p.grad is not None]
print('forward+backward OK in', round(time.time() - t0), 's | loss', round(float(loss), 3), '| LoRA params with grad', len(gn), 'of', sum(1 for p in pm.parameters() if p.requires_grad), '| B grads nonzero:', sum(g > 0 for g in gn))
# export in scorer format
out = Path('/kaggle/working/e4b_adapter_raw'); pm.save_pretrained(str(out))
os.environ['SCORER_MODEL'] = 'google/gemma-4-e4b-it-qat-w4a16-ct'
exec(MAKE_ADAPTERS_SRC.replace('if __name__ == "__main__":', 'if False:'))
rep = main(out, Path('/kaggle/working/e4b_adapters'), None)
print('ADAPTER EXPORT OK:', json.dumps({k: rep[k] for k in ('n_tensors', 'rank', 'alpha')}), '| g1_full tensors', rep['g1_full']['tensors'], 'bytes', rep['g1_full']['bytes'], '| modules', rep['g1_full']['target_modules'])
print(json.loads((Path('/kaggle/working/e4b_adapters/g1_full/adapter_config.json')).read_text())['base_model_name_or_path'])
print('E4B DRY RUN: PASS')